# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.


# 6_query_routing_experiments.ipynb
PURPOSE

Evaluate different query routing strategies for ARIA-Lite GraphRAG.

We compare:

Rule-based routing
LLM-based routing
Hybrid routing

The notebook creates a comparison table:

| Query | Ground Truth | Rule-Based | LLM-Based | Hybrid |

This helps analyze:

routing quality
ambiguous queries
failure cases
strengths/weaknesses of each approach

In [ ]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================

!pip install transformers accelerate bitsandbytes pandas
!pip install -U spacy
!pip install -U scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 56.0 MB/s eta 0:00:00
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5
    Uninstalling confection-0.1.5:
      Successfully uninstalled confection-0.1.5
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: thinc
    Found existing installation: thinc 8.2.5
    Uninstalling thinc-8.2.5:
      Successfully uninstalled thinc-8.2.5
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.3
    Uninstalling weasel-0.4.3:
      Successfully uninstalled weasel-0.4.3
  Attempting uninstall: spacy
    Found existing installation: spacy 3.7

In [ ]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

import pandas as pd
from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

In [ ]:
# ============================================================
# SECTION 3 — Load Lightweight LLM
# ============================================================

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("LLM loaded")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

LLM loaded


In [ ]:
# ============================================================
# SECTION 4 — Example Queries + Ground Truth
# ============================================================

"""
LOCAL:

focused
mechanistic
entity-centric
narrow biomedical reasoning

GLOBAL:

broad
overview
survey
multi-theme
"""

# ============================================================
# SECTION 4 — Example Queries + Ground Truth (EXPANDED)
# ============================================================

"""
LOCAL:
- focused
- mechanistic
- entity-centric
- narrow biomedical reasoning
- asks "how/why" for specific interactions
- 2+ specific entities mentioned

GLOBAL:
- broad
- overview
- survey
- multi-theme
- asks "what are", "summarize", "compare"
- thematic exploration
"""

queries = [
    # Original 8 queries
    {
        "query": "How does trastuzumab target HER2 in breast cancer?",
        "gt": "LOCAL"
    },
    {
        "query": "What are the major machine learning methods used in breast cancer diagnosis?",
        "gt": "GLOBAL"
    },
    {
        "query": "Explain the role of PD-L1 in tumor progression.",
        "gt": "LOCAL"
    },
    {
        "query": "What are recent trends in AI for breast cancer imaging?",
        "gt": "GLOBAL"
    },
    {
        "query": "How does XGBoost predict breast cancer survival?",
        "gt": "LOCAL"
    },
    {
        "query": "Compare deep learning and classical ML approaches for breast cancer.",
        "gt": "GLOBAL"
    },
    {
        "query": "What biomarkers are associated with HER2-positive tumors?",
        "gt": "LOCAL"
    },
    {
        "query": "Summarize multi-omics approaches in breast cancer research.",
        "gt": "GLOBAL"
    },

    # ===== NEW TEST CASES =====

    # LOCAL: Specific mechanism queries
    {
        "query": "What is the relationship between EGFR and MAPK signaling?",
        "gt": "LOCAL"
    },
    {
        "query": "How does Random Forest handle feature selection in genomic data?",
        "gt": "LOCAL"
    },
    {
        "query": "Describe the interaction between CD21 and dendritic cells.",
        "gt": "LOCAL"
    },
    {
        "query": "Why does tumor diameter correlate with lymph node metastasis?",
        "gt": "LOCAL"
    },
    {
        "query": "How is ROC AUC calculated for classification models?",
        "gt": "LOCAL"
    },

    # LOCAL: Entity-pair relationships
    {
        "query": "CNN for mammography image classification",
        "gt": "LOCAL"
    },
    {
        "query": "SHAP values in breast cancer prediction models",
        "gt": "LOCAL"
    },
    {
        "query": "Recursive feature elimination with cross-validation",
        "gt": "LOCAL"
    },

    # GLOBAL: Overview/landscape queries
    {
        "query": "Give me an overview of immune response research in breast cancer.",
        "gt": "GLOBAL"
    },
    {
        "query": "What types of biomarkers are studied in luminal breast cancer?",
        "gt": "GLOBAL"
    },
    {
        "query": "Summarize the state of explainable AI in oncology.",
        "gt": "GLOBAL"
    },
    {
        "query": "What are common feature selection techniques?",
        "gt": "GLOBAL"
    },
    {
        "query": "List imaging modalities used in breast cancer diagnosis.",
        "gt": "GLOBAL"
    },

    # GLOBAL: Comparison/survey queries
    {
        "query": "Compare supervised and unsupervised learning for cancer subtyping.",
        "gt": "GLOBAL"
    },
    {
        "query": "What are the differences between Random Forest and XGBoost?",
        "gt": "GLOBAL"
    },
    {
        "query": "Overview of interpretability methods in medical AI.",
        "gt": "GLOBAL"
    },

    # EDGE CASES: Ambiguous queries (test router robustness)
    {
        "query": "HER2",  # Single entity, no context
        "gt": "LOCAL"  # Or GLOBAL depending on router - you decide threshold
    },
    {
        "query": "Machine learning",  # Very broad single entity
        "gt": "GLOBAL"
    },
    {
        "query": "Tell me about breast cancer research.",  # Very vague
        "gt": "GLOBAL"
    },
    {
        "query": "trastuzumab efficacy",  # 1-2 entities, somewhat specific
        "gt": "LOCAL"
    },
    {
        "query": "What papers mention AI and imaging?",  # Meta-query about papers
        "gt": "GLOBAL"
    },

    # Implicit intent queries (test keyword detection)
    {
        "query": "I want to learn about methods for predicting metastasis.",
        "gt": "GLOBAL"  # "methods" plural = overview
    },
    {
        "query": "Show me how support vector machines work for classification.",
        "gt": "LOCAL"  # "how X works" = mechanistic
    },
    {
        "query": "Landscape of targeted therapies in HER2+ breast cancer.",
        "gt": "GLOBAL"  # "landscape" = overview trigger
    }
]

In [ ]:
# ============================================================
# SECTION 5 — Rule-Based Router (IMPROVED)
# ============================================================

# TIER 1: Strong GLOBAL triggers (highest priority)
GLOBAL_STRONG = [
    "overview", "summarize", "summary", "landscape",
    "what are", "what types", "types of",
    "compare", "comparison", "differences between",
    "list", "survey", "state of"
]

# TIER 2: Weaker GLOBAL signals
GLOBAL_WEAK = [
    "approaches", "methods used", "trends", "recent trends",
    "major", "common"
]

# TIER 3: Strong LOCAL triggers
LOCAL_STRONG = [
    "how does", "why does", "how is",
    "mechanism", "interaction between",
    "relationship between", "role of",
    "explain the role", "describe the interaction"
]

# TIER 4: Weaker LOCAL signals
LOCAL_WEAK = [
    "predict", "target", "associated", "explain"
]


def rule_based_router(query):
    q_lower = query.lower()

    # TIER 1: Check strong GLOBAL triggers first
    for trigger in GLOBAL_STRONG:
        if trigger in q_lower:
            return "GLOBAL"

    # TIER 2: Check strong LOCAL triggers
    for trigger in LOCAL_STRONG:
        if trigger in q_lower:
            return "LOCAL"

    # TIER 3: Score weaker signals
    global_score = sum(1 for kw in GLOBAL_WEAK if kw in q_lower)
    local_score = sum(1 for kw in LOCAL_WEAK if kw in q_lower)

    if global_score > local_score:
        return "GLOBAL"
    elif local_score > global_score:
        return "LOCAL"

    # TIER 4: Fallback - use query length heuristic
    # Very short queries (1-3 words) = entity lookup = LOCAL
    # Longer queries without clear signals = GLOBAL (exploratory)
    word_count = len(query.split())
    if word_count <= 3:
        return "LOCAL"
    else:
        return "GLOBAL"

In [ ]:
# ============================================================
# SECTION 6 — LLM-Based Router
# ============================================================

def llm_router(query):

  prompt = f"""You are a classifier.

Return ONLY one of these two words:

LOCAL
GLOBAL

No explanations. No punctuation. No extra text.

Query: {query}

Query: How does HER2 interact with trastuzumab?
Answer: LOCAL

Query: What are trends in breast cancer imaging?
Answer: GLOBAL

Query: Explain EGFR signaling mechanism
Answer: LOCAL

Query: Overview of machine learning in cancer diagnosis
Answer: GLOBAL

Think step-by-step:
1. Does the query contain GLOBAL keywords (what are, summarize, compare, list, overview, trends)?
2. Does it ask about a specific mechanism/interaction between named entities?
3. Based on 1 and 2, what is the classification?

Format your response:
Answer: LOCAL or GLOBAL
"""

  output = llm(prompt, max_new_tokens=5, do_sample=False)

  text = output[0]["generated_text"]

  response = text[len(prompt):].strip().upper()

  print(f"Query: {query}")
  print(f"Response: {response}\n\n\n\n")

  if "GLOBAL" in response:
    return "GLOBAL"

  return "LOCAL"

In [ ]:
# ============================================================
# SECTION 6.5 — Entity Extraction Helper
# ============================================================

import spacy

# Load your Scispacy model (same one you used for papers)
nlp = spacy.load("en_core_sci_md")

def extract_query_entities(query):
    """
    Extract entities from query using Scispacy NER.
    Returns list of entity texts.
    """
    doc = nlp(query)

    entities = []
    for ent in doc.ents:
        entities.append(ent.text.lower())

    return entities

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [ ]:
# ============================================================
# SECTION 7 — Multi-level Router
# ============================================================

def hybrid_router(query):
    q_lower = query.lower()

    # --------------------------------------------------------
    # TIER 1: Check strong phrase-level triggers FIRST
    # --------------------------------------------------------

    # Strong GLOBAL triggers (immediate return)
    for trigger in GLOBAL_STRONG:
        if trigger in q_lower:
            return "GLOBAL"

    # Strong LOCAL triggers (immediate return)
    for trigger in LOCAL_STRONG:
        if trigger in q_lower:
            return "LOCAL"

    # --------------------------------------------------------
    # TIER 2: Score weaker signals
    # --------------------------------------------------------

    global_score = sum(1 for kw in GLOBAL_WEAK if kw in q_lower)
    local_score = sum(1 for kw in LOCAL_WEAK if kw in q_lower)

    # High confidence from weak signals
    if abs(global_score - local_score) >= 2:
        if global_score > local_score:
            return "GLOBAL"
        return "LOCAL"

    # Clear winner from weak signals
    if global_score > local_score:
        return "GLOBAL"
    elif local_score > global_score:
        return "LOCAL"

    # --------------------------------------------------------
    # TIER 3: Entity-based routing
    # --------------------------------------------------------

    # Extract entities from query
    entities = extract_query_entities(query)

    if len(entities) >= 2:
        return "LOCAL"  # Multi-entity = specific relationship
    elif len(entities) == 0:
        return "GLOBAL"  # No entities = vague overview

    # --------------------------------------------------------
    # TIER 4: Query length heuristic fallback
    # --------------------------------------------------------

    word_count = len(query.split())

    if word_count <= 3:
        return "LOCAL"   # Very short = entity lookup
    else:
        return "GLOBAL"  # Longer ambiguous = exploratory


In [ ]:
# ============================================================
# SECTION 8 — Evaluate Routing Methods
# ============================================================

results = []

for item in queries:

    query = item["query"]
    gt = item["gt"]

    rb = rule_based_router(query)
    llm_pred = llm_router(query)
    hybrid = hybrid_router(query)

    results.append({
        "Query": query,
        "GT": gt,
        "Rule-Based": rb,
        "LLM-Based": llm_pred,
        "Hybrid": hybrid
    })

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: How does trastuzumab target HER2 in breast cancer?
Response: QUERY: WHAT ARE






Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What are the major machine learning methods used in breast cancer diagnosis?
Response: QUERY: WHAT ARE






Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Query: What papers mention AI and imaging?
Response: QUERY: WHAT ARE






Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: I want to learn about methods for predicting metastasis.
Response: QUERY: WHAT ARE






Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Show me how support vector machines work for classification.
Response: QUERY: DISCUSS




Query: Landscape of targeted therapies in HER2+ breast cancer.
Response: QUERY: WHAT ARE






In [ ]:
# ============================================================
# SECTION 9 — Display Results Table
# ============================================================

results_df = pd.DataFrame(results)

results_df

,Query,GT,Rule-Based,LLM-Based,Hybrid
0,How does trastuzumab target HER2 in breast can...,LOCAL,LOCAL,LOCAL,LOCAL
1,What are the major machine learning methods us...,GLOBAL,GLOBAL,LOCAL,GLOBAL
2,Explain the role of PD-L1 in tumor progression.,LOCAL,LOCAL,LOCAL,LOCAL
3,What are recent trends in AI for breast cancer...,GLOBAL,GLOBAL,LOCAL,GLOBAL
4,How does XGBoost predict breast cancer survival?,LOCAL,LOCAL,LOCAL,LOCAL
5,Compare deep learning and classical ML approac...,GLOBAL,GLOBAL,LOCAL,GLOBAL
6,What biomarkers are associated with HER2-posit...,LOCAL,LOCAL,LOCAL,LOCAL
7,Summarize multi-omics approaches in breast can...,GLOBAL,GLOBAL,LOCAL,GLOBAL
8,What is the relationship between EGFR and MAPK...,LOCAL,LOCAL,LOCAL,LOCAL
9,How does Random Forest handle feature selectio...,LOCAL,LOCAL,LOCAL,LOCAL


In [ ]:
# ============================================================
# SECTION 10 — Accuracy Comparison
# ============================================================

rb_acc = (results_df["GT"] == results_df["Rule-Based"]).mean()
llm_acc = (results_df["GT"] == results_df["LLM-Based"]).mean()
hybrid_acc = (results_df["GT"] == results_df["Hybrid"]).mean()

print("\n============================")
print("ROUTING ACCURACY")
print("============================")

print(f"Rule-Based : {rb_acc:.2f}")
print(f"LLM-Based  : {llm_acc:.2f}")
print(f"Hybrid     : {hybrid_acc:.2f}")


ROUTING ACCURACY
Rule-Based : 0.84
LLM-Based  : 0.47
Hybrid     : 0.88


In [ ]:
# ============================================================
# SECTION 11 — Analyze Failure Cases
# ============================================================

for idx, row in results_df.iterrows():

    print("\n================================================")
    print("QUERY:")
    print(row["Query"])

    print("\nGround Truth:", row["GT"])
    print("Rule-Based :", row["Rule-Based"])
    print("LLM-Based  :", row["LLM-Based"])
    print("Hybrid     :", row["Hybrid"])


QUERY:
How does trastuzumab target HER2 in breast cancer?

Ground Truth: LOCAL
Rule-Based : LOCAL
LLM-Based  : LOCAL
Hybrid     : LOCAL

QUERY:
What are the major machine learning methods used in breast cancer diagnosis?

Ground Truth: GLOBAL
Rule-Based : GLOBAL
LLM-Based  : LOCAL
Hybrid     : GLOBAL

QUERY:
Explain the role of PD-L1 in tumor progression.

Ground Truth: LOCAL
Rule-Based : LOCAL
LLM-Based  : LOCAL
Hybrid     : LOCAL

QUERY:
What are recent trends in AI for breast cancer imaging?

Ground Truth: GLOBAL
Rule-Based : GLOBAL
LLM-Based  : LOCAL
Hybrid     : GLOBAL

QUERY:
How does XGBoost predict breast cancer survival?

Ground Truth: LOCAL
Rule-Based : LOCAL
LLM-Based  : LOCAL
Hybrid     : LOCAL

QUERY:
Compare deep learning and classical ML approaches for breast cancer.

Ground Truth: GLOBAL
Rule-Based : GLOBAL
LLM-Based  : LOCAL
Hybrid     : GLOBAL

QUERY:
What biomarkers are associated with HER2-positive tumors?

Ground Truth: LOCAL
Rule-Based : LOCAL
LLM-Based  : LOCAL
